In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

# from sklearn.linear_model import LinearRegression, Ridge, Lasso
# from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
# from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
# from xgboost import XGBRegressor
# from lightgbm import LGBMRegressor
from copy import deepcopy


from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('/kaggle/input/nifty-50-25-yrs-data/data.csv')
df.head()

In [ ]:
def return_pairs(column, days):
    pricess = list(column)
    X = []
    y = []
    for i in range(len(pricess) - days):
        X.append(pricess[i:i+days])
        y.append(pricess[i+days])
    return np.array(X), np.array(y)

target_columns =  ['High']
day_chunks =  [30, 60, 90]

chunked_data = {}

for col in target_columns:
    for days in day_chunks:
        key_X = f"X_{col}_{days}"
        key_y = f"y_{col}_{days}"
        X, y = return_pairs(df[col], days)
        chunked_data[key_X] = X
        chunked_data[key_y] = y


chunk_pairs = []

for key in chunked_data.keys():
    if key.startswith("X_"):
        y_key = key.replace("X_", "y_")
        if y_key in chunked_data:
            chunk_pairs.append([key, y_key])

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, GRU, Bidirectional


def build_rnn(input_shape):
    model = Sequential([
        SimpleRNN(50, activation='tanh', input_shape=input_shape),
        Dense(1)   # regression output
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

def build_lstm(input_shape):
    model = Sequential([
        LSTM(50, activation='tanh', input_shape=input_shape),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

def build_gru(input_shape):
    model = Sequential([
        GRU(50, activation='tanh', input_shape=input_shape),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

def build_bilstm(input_shape):
    model = Sequential([
        Bidirectional(LSTM(50, activation='tanh'), input_shape=input_shape),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
ml_models = [
    ("KNN", KNeighborsRegressor())
]

dl_models = {
    "RNN": build_rnn,
    "LSTM": build_lstm,
    "GRU": build_gru,
    "Bidirectional_LSTM": build_bilstm
}

In [ ]:
trained_models = {}

for X, y in tqdm(chunk_pairs):
    X_data = chunked_data[X]
    y_data = chunked_data[y]

    X_train, X_test, y_train, y_test = train_test_split(
        X_data, y_data, test_size=0.1, random_state=42
    )

    # ML models
    for model_name, model in tqdm(ml_models):
        key = model_name + '_' + X[2:]
        model_copy = deepcopy(model)
        model_copy.fit(X_train, y_train)

        y_train_pred = model_copy.predict(X_train)
        y_test_pred = model_copy.predict(X_test)

        trained_models[key] = {
            'model': model_copy,
            'train_mae': mean_absolute_error(y_train, y_train_pred),
            'train_rmse': np.sqrt(mean_squared_error(y_train, y_train_pred)),
            'test_mae': mean_absolute_error(y_test, y_test_pred),
            'test_rmse': np.sqrt(mean_squared_error(y_test, y_test_pred))
        }

    # DL models
    X_train_rnn = np.expand_dims(X_train, -1)
    X_test_rnn = np.expand_dims(X_test, -1)

    for model_name, builder in tqdm(dl_models.items()):
        key = model_name + '_' + X[2:]
        model_dl = builder((X_train.shape[1], 1))

        model_dl.fit(X_train_rnn, y_train, epochs=50, batch_size=8, verbose=0)

        y_train_pred = model_dl.predict(X_train_rnn).flatten()
        y_test_pred = model_dl.predict(X_test_rnn).flatten()

        trained_models[key] = {
            'model': model_dl,
            'train_mae': mean_absolute_error(y_train, y_train_pred),
            'train_rmse': np.sqrt(mean_squared_error(y_train, y_train_pred)),
            'test_mae': mean_absolute_error(y_test, y_test_pred),
            'test_rmse': np.sqrt(mean_squared_error(y_test, y_test_pred))
        }


In [ ]:
results_df = pd.DataFrame([
    {"Model": name, **metrics}
    for name, metrics in trained_models.items()])

results_df.sort_values(by = 'test_mae', ascending = True).head(50)

Witout scaling, KNN worked the best.

In [ ]:
from matplotlib import pyplot as plt

# Filtering
top_model = results_df.sort_values(by='test_mae', ascending=True).head()
model_types = pd.Series([i.split('_')[0] for i in top_model['Model']])
model_counts = model_types.value_counts().sort_values(ascending=False)

# Plotting
plt.figure(figsize=(10, 5))
plt.bar(model_counts.index, model_counts.values)

# Labels and aesthetics
plt.xlabel('Model Type')
plt.ylabel('Top Model')
plt.title('Model Type Frequency Among Models (Lowest Test MAE)')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

trained_models = {}

for X, y in tqdm(chunk_pairs):
    X_data = chunked_data[X]
    y_data = chunked_data[y]

    # ---------------- ML models ----------------
    for model_name, model in tqdm(ml_models):
        key = model_name + '_' + X[2:]
        model_copy = deepcopy(model)
        model_copy.fit(X_data, y_data)

        y_train_pred = model_copy.predict(X_data)
        y_test_pred = model_copy.predict(
            chunked_data[X][int(0.9*len(X_data)):]
        )

        trained_models[key] = {
            'model': model_copy,
            'train_mae': mean_absolute_error(y_data[:-len(y_test_pred)], y_train_pred[:-len(y_test_pred)]),
            'train_rmse': np.sqrt(mean_squared_error(y_data[:-len(y_test_pred)], y_train_pred[:-len(y_test_pred)])),
            'test_mae': mean_absolute_error(y_data[-len(y_test_pred):], y_test_pred),
            'test_rmse': np.sqrt(mean_squared_error(y_data[-len(y_test_pred):], y_test_pred))
        }

    # ---------------- DL models (with scaling) ----------------
    # Separate scalers for X and y
    X_scaler = MinMaxScaler(feature_range=(0, 1))
    y_scaler = MinMaxScaler(feature_range=(0, 1))

    # Scale
    X_scaled = X_scaler.fit_transform(X_data)
    y_scaled = y_scaler.fit_transform(y_data.reshape(-1, 1)).flatten()

    # Train-test split
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
        X_scaled, y_scaled, test_size=0.1, random_state=42
    )

    # Reshape for RNN/LSTM/GRU (samples, timesteps, features=1)
    X_train_rnn = np.expand_dims(X_train_s, -1)
    X_test_rnn = np.expand_dims(X_test_s, -1)

    for model_name, builder in tqdm(dl_models.items()):
        key = model_name + '_' + X[2:]
        model_dl = builder((X_train_rnn.shape[1], 1))

        # Train for 50 epochs
        model_dl.fit(X_train_rnn, y_train_s, epochs=50, batch_size=8, verbose=0)

        # Predictions (scaled)
        y_train_pred_s = model_dl.predict(X_train_rnn).flatten()
        y_test_pred_s = model_dl.predict(X_test_rnn).flatten()

        # Inverse scale back to original price
        y_train_pred = y_scaler.inverse_transform(y_train_pred_s.reshape(-1, 1)).flatten()
        y_test_pred = y_scaler.inverse_transform(y_test_pred_s.reshape(-1, 1)).flatten()

        y_train_orig = y_scaler.inverse_transform(y_train_s.reshape(-1, 1)).flatten()
        y_test_orig = y_scaler.inverse_transform(y_test_s.reshape(-1, 1)).flatten()

        trained_models[key] = {
            'model': model_dl,
            'train_mae': mean_absolute_error(y_train_orig, y_train_pred),
            'train_rmse': np.sqrt(mean_squared_error(y_train_orig, y_train_pred)),
            'test_mae': mean_absolute_error(y_test_orig, y_test_pred),
            'test_rmse': np.sqrt(mean_squared_error(y_test_orig, y_test_pred))
        }

In [ ]:
results_df = pd.DataFrame([
    {"Model": name, **metrics}
    for name, metrics in trained_models.items()])

results_df.sort_values(by = 'test_mae', ascending = True).head(50)

This result came after scaling the data for DL Models. LSTM performed best with lowest test mean absolute error.